# 06. Chronos-Bolt на национальных рядах СберИндекса

Zero-shot прогноз пяти национальных рядов расходов с историей с декабря 2018 года.
Отсечки — январь 2023 — сентябрь 2025; горизонт — 12 месяцев. На каждой отсечке доступна только предшествующая история.

Вход: национальные расходы из `data/raw/`, имя файла задано в `configs/panel.yaml`.
Результат: `results/chronos_national.parquet`. GPU используется при наличии, иначе CPU.

In [ ]:
from pathlib import Path
import yaml
import numpy as np
import pandas as pd
import torch
from chronos import BaseChronosPipeline

MODEL = "amazon/chronos-bolt-base"
CUTOFFS = pd.date_range("2023-01-01", "2025-09-01", freq="MS")
PRED_LEN, QUANTILES = 12, [0.1, 0.5, 0.9]
device = "cuda" if torch.cuda.is_available() else "cpu"

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
OUT = ROOT / "results"; OUT.mkdir(exist_ok=True)
CFG = yaml.safe_load((ROOT / "configs" / "panel.yaml").read_text(encoding="utf-8"))
sp = pd.read_parquet(ROOT / "data" / "raw" / CFG["files"]["sber_spend"])
sp["date"] = pd.to_datetime(sp.period)
y = sp.pivot_table(index="date", columns="type", values="value").sort_index()
print(y.shape, y.index.min().date(), y.index.max().date())

pipe = BaseChronosPipeline.from_pretrained(MODEL, device_map=device,
                                           torch_dtype=torch.bfloat16 if device == "cuda" else torch.float32)

In [ ]:
rows = []
for T in CUTOFFS:
    hist = y[y.index <= T]
    ctx = [torch.tensor(hist[c].dropna().values, dtype=torch.float32) for c in y.columns]
    q, _ = pipe.predict_quantiles(ctx, prediction_length=PRED_LEN, quantile_levels=QUANTILES)
    q = q.float().cpu().numpy()
    ds = pd.date_range(T + pd.DateOffset(months=1), periods=PRED_LEN, freq="MS")
    for j, c in enumerate(y.columns):
        rows.append(pd.DataFrame({"series": c, "origin": T, "ds": ds,
                                  "q10": q[j, :, 0], "yhat": q[j, :, 1], "q90": q[j, :, 2]}))
fc = pd.concat(rows, ignore_index=True)
fc.to_parquet(OUT / "chronos_national.parquet", index=False)
print(fc.shape, OUT / "chronos_national.parquet")